# Join Integrity

Validate cross-sheet joins, diagnose unmatched rows, and export the enriched task dataset.

In [1]:
from pathlib import Path
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

ROOT = Path.cwd().resolve()
while not (ROOT / "agents.md").exists():
    if ROOT.parent == ROOT:
        raise RuntimeError("Could not locate project root from notebook directory.")
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

warnings.filterwarnings("ignore")
pd.options.display.max_columns = 120
pd.options.display.max_rows = 50

from data_analysis._shared.eda_utils import (
    ENRICHED_PATH,
    FEATURE_GROUPS,
    NUMERIC_OUTLIER_COLUMNS,
    build_enriched_tasks,
    categorical_variants,
    cross_sheet_consistency_report,
    detect_outliers,
    diagnose_client_unmatched,
    diagnose_pair_unmatched,
    diagnose_schedule_unmatched,
    dtype_audit,
    exact_duplicates,
    join_datasets,
    load_all_data,
    load_enriched_tasks,
    missing_summary,
    missingness_by_group,
    missingness_correlation,
    near_duplicates_data,
    proofread_reviewer_comparison,
    same_language_examples,
    same_language_report,
    set_plot_style,
    sheet_dimensions_report,
    summarize_join_integrity,
    timestamp_logic_report,
    timestamp_violation_samples,
    translator_on_time_summary,
    translator_quality_summary,
    wilson_interval,
    zero_value_report,
)

set_plot_style()
sheets = load_all_data()
data = sheets["data"]
schedules = sheets["schedules"]
clients = sheets["clients"]
pairs = sheets["pairs"]
print(f"Loaded Data={data.shape}, Schedules={schedules.shape}, Clients={clients.shape}, Pairs={pairs.shape}")

Loaded Data=(997933, 29), Schedules=(983, 11), Clients=(2645, 6), Pairs=(4688, 7)


In [2]:
joined = join_datasets()
summary = summarize_join_integrity(joined)
join_summary = pd.DataFrame(
    [
        {"join": "Data -> Clients", "match_rate_pct": round(summary.data_to_clients_match_rate * 100, 3)},
        {"join": "Data -> TranslatorsCost+Pairs", "match_rate_pct": round(summary.data_to_pairs_match_rate * 100, 3)},
        {"join": "Data -> Schedules", "match_rate_pct": round(summary.data_to_schedules_match_rate * 100, 3)},
    ]
)
display(join_summary)

,join,match_rate_pct
0,Data -> Clients,100.000
1,Data -> TranslatorsCost+Pairs,99.999
2,Data -> Schedules,100.000


In [3]:
client_diag = diagnose_client_unmatched(joined, clients)
pair_diag = diagnose_pair_unmatched(joined, pairs)
schedule_diag = diagnose_schedule_unmatched(joined, schedules)

display(client_diag["diagnosis"].value_counts(dropna=False).rename_axis("diagnosis").reset_index(name="rows"))
display(pair_diag["diagnosis"].value_counts(dropna=False).rename_axis("diagnosis").reset_index(name="rows"))
display(schedule_diag["diagnosis"].value_counts(dropna=False).rename_axis("diagnosis").reset_index(name="rows"))

display(client_diag.head(30))
display(pair_diag.head(30))
display(schedule_diag.head(30))

,diagnosis,rows


,diagnosis,rows
0,language_or_name_spelling_variant,5


,diagnosis,rows


,MANUFACTURER,diagnosis


,TRANSLATOR,SOURCE_LANG,TARGET_LANG,TRANSLATOR_KEY,SOURCE_LANG_CLEAN,TARGET_LANG_CLEAN,diagnosis
0,Acolmiztli,Spanish (LA),Nahuatl,acolmiztli,spanish (la),nahuatl,language_or_name_spelling_variant
1,Citlali,English,Nahuatl,citlali,english,nahuatl,language_or_name_spelling_variant
2,Fiamma Baldomero,Finnish,Spanish (Iberian),fiamma baldomero,finnish,spanish (iberian),language_or_name_spelling_variant
3,Octavi,Finnish,Finnish,octavi,finnish,finnish,language_or_name_spelling_variant
4,Ryan,Finnish,Spanish (Iberian),ryan,finnish,spanish (iberian),language_or_name_spelling_variant


,TRANSLATOR,diagnosis


In [4]:
enriched = build_enriched_tasks(save=True)
print(f"Saved enriched dataset to: {ENRICHED_PATH}")
print(f"Enriched dataset shape: {enriched.shape}")

key_columns = []
for cols in FEATURE_GROUPS.values():
    key_columns.extend(cols)
key_columns = list(dict.fromkeys(key_columns + ["ON_TIME", "HIGH_QUALITY_8", "SELLING_HOURLY_PRICE", "MIN_QUALITY"]))

completeness = (
    enriched[key_columns]
    .isna()
    .mean()
    .mul(100)
    .round(2)
    .rename("missing_pct")
    .reset_index()
    .rename(columns={"index": "column"})
    .sort_values("missing_pct", ascending=False)
)
display(completeness)

Saved enriched dataset to: C:\Users\herme\UNI\2\SynthesisProject\SytnthesisProject\data\interim\enriched_tasks.csv
Enriched dataset shape: (997933, 80)


,column,missing_pct
26,SHIFT_OVERLAP_PCT,0.50
8,PRIOR_QUALITY_MEAN_PAIR,0.47
15,PUNCTUALITY_TREND_DELTA,0.39
9,PRIOR_QUALITY_MEAN_TASK_TYPE,0.29
12,PRIOR_ON_TIME_RATE_TASK_TYPE,0.29
7,ROLLING_QUALITY_STD_5,0.18
11,PRIOR_ON_TIME_RATE_TRANSLATOR,0.10
5,PRIOR_QUALITY_MEAN_TRANSLATOR,0.10
6,ROLLING_QUALITY_MEAN_5,0.10
13,PRIOR_AVG_LATENESS_MINUTES,0.10


In [5]:
final_gap_summary = pd.DataFrame(
    [
        {
            "gap_area": "Client join gaps",
            "rows_pct": round((~joined["CLIENT_MATCH_EXACT"]).mean() * 100, 3),
        },
        {
            "gap_area": "Pair join gaps",
            "rows_pct": round((~joined["PAIR_MATCH_EXACT"]).mean() * 100, 3),
        },
        {
            "gap_area": "Schedule join gaps",
            "rows_pct": round((~joined["SCHEDULE_MATCH_EXACT"]).mean() * 100, 3),
        },
        {
            "gap_area": "Approximate availability rows",
            "rows_pct": round((enriched["AVAILABILITY_FEATURE_RELIABILITY"] == "approximate_night_shift").mean() * 100, 3),
        },
    ]
)
display(final_gap_summary)

,gap_area,rows_pct
0,Client join gaps,0.000
1,Pair join gaps,0.001
2,Schedule join gaps,0.000
3,Approximate availability rows,0.003
